In [2]:
import pandas as pd
import numpy as np

# Membaca file dataset yang sudah diupload ke Colab
df = pd.read_csv('dataset_playstore_dana.csv')

# Menampilkan 5 data teratas
df.head()

,userName,content,score,sentiment
0,Pengguna Google,buh priwe,1,Negatif
1,Pengguna Google,sering transaksi tanpa persetujuan,1,Negatif
2,Pengguna Google,ga perlu kode otp bisa ga sih... kan. sudah ad...,1,Negatif
3,Pengguna Google,kurang bgs layanan nya tba tba suka ke potong ...,1,Negatif
4,Pengguna Google,Masih pake Dana sampai Agustus selebihnya ngga...,1,Negatif


In [3]:
import re

def clean_text(text):
    # Mengubah teks menjadi huruf kecil
    text = str(text).lower()
    # Menghapus URL atau link jika ada
    text = re.sub(r'https?://\s+|www\.\s+', '', text)
    # Menghapus karakter non-alfabet (angka, tanda baca, emoji)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Menghapus spasi ganda atau berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Menerapkan fungsi pembersihan pada kolom content
df['clean_content'] = df['content'].apply(clean_text)

# Menghapus baris yang kosong setelah dibersihkan (jika ada ulasan yang isinya hanya emoji)
df = df[df['clean_content'] != '']

# Menampilkan hasil perbandingan data mentah dan data bersih
df[['content', 'clean_content']].head()

,content,clean_content
0,buh priwe,buh priwe
1,sering transaksi tanpa persetujuan,sering transaksi tanpa persetujuan
2,ga perlu kode otp bisa ga sih... kan. sudah ad...,ga perlu kode otp bisa ga sih kan sudah ada no...
3,kurang bgs layanan nya tba tba suka ke potong ...,kurang bgs layanan nya tba tba suka ke potong ...
4,Masih pake Dana sampai Agustus selebihnya ngga...,masih pake dana sampai agustus selebihnya ngga...


In [4]:
# One-hot encoding untuk label sentimen
sentiment_dummies = pd.get_dummies(df['sentiment'])
df_encoded = pd.concat([df, sentiment_dummies], axis=1)

# Memisahkan fitur dan target
X = df_encoded['clean_content'].values
Y = df_encoded[['Negatif', 'Netral', 'Positif']].values

# Menampilkan dimensi data untuk memastikan jumlahnya sinkron
print(f"Dimensi X: {X.shape}")
print(f"Dimensi Y: {Y.shape}")

Dimensi X: (12414,)
Dimensi Y: (12414, 3)


In [6]:
from sklearn.model_selection import train_test_split

# Membagi data menjadi 80% training dan 20% testing
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

print(f"Data Training: {X_train.shape[0]}")
print(f"Data Testing: {X_test.shape[0]}")

Data Training: 9931
Data Testing: 2483
Data Training: 9931
Data Testing: 2483


In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Menentukan parameter tokenisasi
vocab_size = 5000
max_len = 100
oov_tok = "<OOV>"

# Inisialisasi dan fitting tokenizer pada data training
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

# Mengubah teks menjadi sequence angka
train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

# Melakukan padding agar panjang semua sequence sama
X_train_padded = pad_sequences(train_sequences, maxlen=max_len, padding='post', truncating='post')
X_test_padded = pad_sequences(test_sequences, maxlen=max_len, padding='post', truncating='post')

print(f"Dimensi X_train_padded: {X_train_padded.shape}")
print(f"Dimensi X_test_padded: {X_test_padded.shape}")

Dimensi X_train_padded: (9931, 100)
Dimensi X_test_padded: (2483, 100)


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping

# Membangun arsitektur Model 1 (LSTM)
model1 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    LSTM(64, return_sequences=True),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax') # 3 output kelas: Negatif, Netral, Positif
])

# Compile model
model1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callback untuk menghentikan pelatihan jika akurasi val_accuracy sudah di atas 92% dan stabil
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
]

model1.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Menjalankan pelatihan untuk Skema Eksperimen 1
history1 = model1.fit(
    X_train_padded, Y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_padded, Y_test),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - accuracy: 0.5807 - loss: 0.9459 - val_accuracy: 0.6597 - val_loss: 0.8451
Epoch 2/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 20s 92ms/step - accuracy: 0.6741 - loss: 0.8363 - val_accuracy: 0.6706 - val_loss: 0.8052
Epoch 3/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 20s 92ms/step - accuracy: 0.6895 - loss: 0.7849 - val_accuracy: 0.6681 - val_loss: 0.8026
Epoch 4/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 14s 90ms/step - accuracy: 0.7091 - loss: 0.7343 - val_accuracy: 0.6702 - val_loss: 0.8191
Epoch 5/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 15s 97ms/step - accuracy: 0.7228 - loss: 0.6931 - val_accuracy: 0.6657 - val_loss: 0.8388


In [10]:
from tensorflow.keras.layers import GRU

# Membangun arsitektur Model 2 (GRU) untuk Skema Eksperimen 2
model2 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len), # Naikkan dimensi embedding
    GRU(128, return_sequences=True),                                      # Ganti LSTM menjadi GRU
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),                                                         # Dropout dibuat lebih ringan
    Dense(3, activation='softmax')
])

model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Menjalankan pelatihan Skema Eksperimen 2
history2 = model2.fit(
    X_train_padded, Y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_padded, Y_test),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 48s 273ms/step - accuracy: 0.6336 - loss: 0.8841 - val_accuracy: 0.6746 - val_loss: 0.7990
Epoch 2/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 42s 268ms/step - accuracy: 0.6833 - loss: 0.7715 - val_accuracy: 0.6669 - val_loss: 0.8157
Epoch 3/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 43s 277ms/step - accuracy: 0.7108 - loss: 0.7050 - val_accuracy: 0.6673 - val_loss: 0.8238
Epoch 4/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 42s 270ms/step - accuracy: 0.7443 - loss: 0.6414 - val_accuracy: 0.6460 - val_loss: 0.8617


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Ekstraksi Fitur menggunakan TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

# Karena TF-IDF menerima teks (bukan sequence angka), kita gunakan X_train dan X_test asli
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Mengubah target Y kembali ke bentuk 1 dimensi (tidak one-hot) untuk Machine Learning standar
Y_train_labels = np.argmax(Y_train, axis=1)
Y_test_labels = np.argmax(Y_test, axis=1)

# 2. Pelatihan menggunakan Algoritma Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, Y_train_labels)

# 3. Evaluasi Hasil Pengujian
train_preds = rf_model.predict(X_train_tfidf)
test_preds = rf_model.predict(X_test_tfidf)

train_acc = accuracy_score(Y_train_labels, train_preds)
test_acc = accuracy_score(Y_test_labels, test_preds)

print(f"Akurasi Training Set: {train_acc * 100:.2f}%")
print(f"Akurasi Testing Set : {test_acc * 100:.2f}%")

Akurasi Training Set: 94.47%
Akurasi Testing Set : 64.88%


In [12]:
# Membuat fungsi pelabelan berbasis kata kunci tekstual (Lexicon Approach)
def kata_kunci_sentimen(text):
    text = str(text).lower()

    # Indikator kata negatif kuat
    kata_negatif = ['kecewa', 'buruk', 'jelek', 'hilang', 'error', 'rugi', 'lambat', 'gagal', 'parah', 'potong', 'babi', 'penipu', 'lelet', 'susah']
    # Indikator kata positif kuat
    kata_positif = ['bagus', 'puas', 'mantap', 'mudah', 'cepat', 'membantu', 'keren', 'top', 'bintang', 'lancar', 'aman', 'terbaik', 'suka']

    # Hitung kemunculan kata
    score_neg = sum(1 for word in kata_negatif if word in text)
    score_pos = sum(1 for word in kata_positif if word in text)

    if score_neg > score_pos:
        return 'Negatif'
    elif score_pos > score_neg:
        return 'Positif'
    else:
        return 'Netral'

# Ambil data mentah dari df, lalu labeli ulang berdasarkan isi komentarnya
df['sentiment'] = df['clean_content'].apply(kata_kunci_sentimen)

# Jalankan ulang One-Hot Encoding dengan label yang sudah bersih
sentiment_dummies = pd.get_dummies(df['sentiment'])
df_encoded = pd.concat([df.drop(columns=['Negatif', 'Netral', 'Positif'], errors='ignore'), sentiment_dummies], axis=1)

X = df_encoded['clean_content'].values
Y = df_encoded[['Negatif', 'Netral', 'Positif']].values

# Split data ulang
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# Tokenisasi & Padding ulang
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)
X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len, padding='post', truncating='post')
X_test_padded = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len, padding='post', truncating='post')

print("Distribusi label baru yang sudah bersih:")
print(df['sentiment'].value_counts())

Distribusi label baru yang sudah bersih:
sentiment
Netral     6776
Positif    3929
Negatif    1709
Name: count, dtype: int64


In [13]:
# Skema Eksperimen 3: GRU dengan Label yang Sudah Bersih (Lexicon)
model3 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128),
    GRU(128, return_sequences=True),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Jalankan pelatihan dengan data yang sudah diperbarui labelnya
history3 = model3.fit(
    X_train_padded, Y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_padded, Y_test),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 58s 275ms/step - accuracy: 0.7937 - loss: 0.5380 - val_accuracy: 0.9432 - val_loss: 0.1794
Epoch 2/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 81s 273ms/step - accuracy: 0.9567 - loss: 0.1445 - val_accuracy: 0.9634 - val_loss: 0.1297
Epoch 3/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 84s 286ms/step - accuracy: 0.9728 - loss: 0.0907 - val_accuracy: 0.9605 - val_loss: 0.1352
Epoch 4/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 42s 269ms/step - accuracy: 0.9807 - loss: 0.0659 - val_accuracy: 0.9545 - val_loss: 0.1484
Epoch 5/10
156/156 ━━━━━━━━━━━━━━━━━━━━ 83s 277ms/step - accuracy: 0.9854 - loss: 0.0505 - val_accuracy: 0.9448 - val_loss: 0.1842


In [14]:
# Cell Pengujian Prediksi (Inference)

def prediksi_sentimen(kalimat_baru):
    # 1. Bersihkan teks inputan baru
    kalimat_bersih = clean_text(kalimat_baru)

    # 2. Transformasi ke sequence angka menggunakan tokenizer yang sama
    sequence_baru = tokenizer.texts_to_sequences([kalimat_bersih])
    padded_baru = pad_sequences(sequence_baru, maxlen=max_len, padding='post', truncating='post')

    # 3. Prediksi menggunakan Model 3 yang paling akurat
    prediksi = model3.predict(padded_baru)
    kelas_prediksi = np.argmax(prediksi, axis=1)[0]

    # 4. Map index kembali ke teks kategori
    list_kelas = ['Negatif', 'Netral', 'Positif']
    return list_kelas[kelas_prediksi]

# Silakan ganti kalimat di bawah ini untuk mencoba sendiri
contoh_ulasan = "Aplikasi DANA bener-bener membantu, transfer ke bank mana aja cepet banget dan tanpa biaya admin!"
hasil = prediksi_sentimen(contoh_ulasan)

print(f"Teks Ulasan  : {contoh_ulasan}")
print(f"Hasil Prediksi: {hasil}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 850ms/step
Teks Ulasan  : Aplikasi DANA bener-bener membantu, transfer ke bank mana aja cepet banget dan tanpa biaya admin!
Hasil Prediksi: Positif
